# Lib

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import random
import kagglehub
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

import torchvision
from torchvision.models.segmentation import deeplabv3_resnet50

import warnings
warnings.filterwarnings("ignore")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# Config


In [1]:
from src.config import *

# Load Data

In [4]:
path = kagglehub.dataset_download(DATASET)

print("Path to dataset files:", path)

os.listdir(path)

image_root = os.path.join(path, 'Cityscape Dataset', 'leftImg8bit')
mask_root = os.path.join(path, 'Fine Annotations', 'gtFine')

print("Image root:", image_root)
print("Mask root:", mask_root)

100%|██████████| 11.0G/11.0G [02:11<00:00, 90.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/electraawais/cityscape-dataset/versions/2
Image root: /root/.cache/kagglehub/datasets/electraawais/cityscape-dataset/versions/2/Cityscape Dataset/leftImg8bit
Mask root: /root/.cache/kagglehub/datasets/electraawais/cityscape-dataset/versions/2/Fine Annotations/gtFine


## Pair image-mask

In [5]:
def get_cityscapes_pairs(left_root, gt_root, split="train"):
    left_split_path = os.path.join(left_root, split)
    gt_split_path = os.path.join(gt_root, split)

    image_paths = []

    for root, _, files in os.walk(left_split_path):
        for file in files:
            if file.endswith("_leftImg8bit.png"):
                image_paths.append(os.path.join(root, file))
    image_paths.sort()

    trainid_masks = {}
    labelid_masks = {}

    for root, _, files in os.walk(gt_split_path):
        for file in files:
            if file.endswith("_gtFine_labelTrainIds.png"):
                trainid_masks[os.path.basename(file)] = os.path.join(root, file)
            elif file.endswith("_gtFine_labelIds.png"):
                labelid_masks[os.path.basename(file)] = os.path.join(root, file)

    pairs = []

    for img_path_str in image_paths:
        img_name = os.path.basename(img_path_str)

        trainid_name = img_name.replace(
            "_leftImg8bit.png",
            "_gtFine_labelTrainIds.png"
        )

        labelid_name = img_name.replace(
            "_leftImg8bit.png",
            "_gtFine_labelIds.png"
        )

        if trainid_name in trainid_masks:
            pairs.append((img_path_str, trainid_masks[trainid_name], "trainId"))
        elif labelid_name in labelid_masks:
            pairs.append((img_path_str, labelid_masks[labelid_name], "labelId"))
        else:
            print(f"Missing mask for: {img_path_str}")

    return pairs

train_pairs = get_cityscapes_pairs(image_root, mask_root, split="train")
val_pairs = get_cityscapes_pairs(image_root, mask_root, split="val")

print("Train pairs:", len(train_pairs))
print("Val pairs:", len(val_pairs))

print("\nExample pair:")
print(train_pairs[70])

Train pairs: 2975
Val pairs: 500

Example pair:
('/root/.cache/kagglehub/datasets/electraawais/cityscape-dataset/versions/2/Cityscape Dataset/leftImg8bit/train/aachen/aachen_000070_000019_leftImg8bit.png', '/root/.cache/kagglehub/datasets/electraawais/cityscape-dataset/versions/2/Fine Annotations/gtFine/train/aachen/aachen_000070_000019_gtFine_labelIds.png', 'labelId')


In [6]:
train_pairs, test_pairs = train_test_split(
    train_pairs,
    train_size=2500,
    test_size=475,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_pairs))
print("Val:", len(val_pairs))
print("Test:", len(test_pairs))

Train: 2500
Val: 500
Test: 475


## Dataset

In [7]:
def convert_labelid_to_trainid(mask):
    train_mask = np.ones_like(mask, dtype=np.uint8) * IGNORE_INDEX

    for label_id, train_id in LABELID_TO_TRAINID.items():
        train_mask[mask == label_id] = train_id

    return train_mask

In [8]:
train_transform = A.Compose([
    A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH),

    A.HorizontalFlip(p=0.5),

    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.3
    ),

    A.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05,
        p=0.3
    ),

    A.GaussianBlur(
        blur_limit=(3, 5),
        p=0.2
    ),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])


val_transform = A.Compose([
    A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

In [9]:
class CityscapesSegmentationDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path, mask_type = self.pairs[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        image = np.array(image)
        mask = np.array(mask)

        if mask.ndim != 2:
            raise ValueError(f"Mask phải là ảnh grayscale, nhưng nhận shape: {mask.shape}")

        if mask_type == "labelId":
            mask = convert_labelid_to_trainid(mask)

        if self.transform is not None:
            transformed = self.transform(image=image, mask=mask)

            image = transformed["image"]
            mask = transformed["mask"]

        mask = mask.long()

        return {
            "image": image,
            "mask": mask,
            "image_path": str(img_path),
            "mask_path": str(mask_path)
        }

In [10]:
train_dataset = CityscapesSegmentationDataset(
    pairs=train_pairs,
    transform=train_transform
)

val_dataset = CityscapesSegmentationDataset(
    pairs=val_pairs,
    transform=val_transform
)

test_dataset = CityscapesSegmentationDataset(
    pairs=test_pairs,
    transform=val_transform
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 2500
Validation dataset: 500
Test dataset: 475


## DataLoader

In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 1250
Validation batches: 250
Test batches: 238


# Function


## Deeplabv3 Resnet50

In [12]:
def build_deeplabv3_resnet50(num_classes):
    try:
        from torchvision.models.segmentation import DeepLabV3_ResNet50_Weights

        weights = DeepLabV3_ResNet50_Weights.DEFAULT

        model = deeplabv3_resnet50(
            weights=weights,
            aux_loss=True
        )

        print("Loaded pretrained DeepLabV3-ResNet50 weights.")

    except Exception as e:
        print("Could not use new torchvision weights API.")
        print("Fallback to pretrained=True.")
        print("Error:", e)

        model = deeplabv3_resnet50(
            pretrained=True,
            aux_loss=True
        )

    # Main classifier
    model.classifier[-1] = nn.Conv2d(
        in_channels=256,
        out_channels=num_classes,
        kernel_size=1
    )

    # Auxiliary classifier nếu có
    if model.aux_classifier is not None:
        model.aux_classifier[-1] = nn.Conv2d(
            in_channels=256,
            out_channels=num_classes,
            kernel_size=1
        )

    return model

## Segmentation Metrics

In [13]:
class SegmentationMetrics:
    def __init__(self, num_classes, ignore_index=255):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.reset()

    def reset(self):
        self.confusion_matrix = torch.zeros(
            self.num_classes,
            self.num_classes,
            dtype=torch.int64
        )

    def update(self, preds, targets):
        preds = preds.detach().cpu()
        targets = targets.detach().cpu()

        valid_mask = targets != self.ignore_index

        preds = preds[valid_mask]
        targets = targets[valid_mask]

        if targets.numel() == 0:
            return

        indices = targets * self.num_classes + preds

        cm = torch.bincount(
            indices,
            minlength=self.num_classes ** 2
        ).reshape(self.num_classes, self.num_classes)

        self.confusion_matrix += cm

    def compute(self):
        cm = self.confusion_matrix.float()

        tp = torch.diag(cm)
        fp = cm.sum(dim=0) - tp
        fn = cm.sum(dim=1) - tp

        denominator = tp + fp + fn
        iou = tp / denominator.clamp(min=1)

        valid_classes = denominator > 0
        miou = iou[valid_classes].mean().item()

        pixel_acc = tp.sum() / cm.sum().clamp(min=1)
        pixel_acc = pixel_acc.item()

        return {
            "mIoU": miou,
            "Pixel Accuracy": pixel_acc,
            "Class IoU": iou.numpy()
        }

## Train

In [14]:
def train_model(
    model,
    train_loader,
    val_loader,
    device,
    num_classes,
    ignore_index=255,
    epochs=10,
    lr=1e-4,
    weight_decay=1e-4,
    checkpoint=None,
    save_path="deeplabv3_resnet50_best.pth",
):
    model = model.to(device)
    model_name = "DeepLabV3-ResNet50"

    criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    )

    history = {
        "train_loss": [],
        "val_loss": [],
    }

    best_val_loss = float("inf")
    start_epoch = 0

    if checkpoint is not None:
        ckpt = torch.load(checkpoint, map_location=device)

        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])

        history = ckpt["history"]
        best_val_loss = ckpt["best_val_loss"]
        start_epoch = ckpt["epoch"] + 1

        print(f"Resume from epoch {start_epoch}")

    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss = 0.0

        for batch in train_loader:
            images = batch["image"].to(device, non_blocking=True)
            masks = batch["mask"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            logits = outputs["out"]

            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(
                    logits,
                    size=masks.shape[-2:],
                    mode="bilinear",
                    align_corners=False
                )

            loss = criterion(logits, masks)

            if "aux" in outputs and outputs["aux"] is not None:
                aux_logits = outputs["aux"]

                if aux_logits.shape[-2:] != masks.shape[-2:]:
                    aux_logits = F.interpolate(
                        aux_logits,
                        size=masks.shape[-2:],
                        mode="bilinear",
                        align_corners=False
                    )

                aux_loss = criterion(aux_logits, masks)
                loss = loss + 0.4 * aux_loss

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(device, non_blocking=True)
                masks = batch["mask"].to(device, non_blocking=True)

                outputs = model(images)
                logits = outputs["out"]

                if logits.shape[-2:] != masks.shape[-2:]:
                    logits = F.interpolate(
                        logits,
                        size=masks.shape[-2:],
                        mode="bilinear",
                        align_corners=False
                    )

                loss = criterion(logits, masks)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)

        scheduler.step(avg_val_loss)

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"[{model_name}] Epoch {epoch + 1:02d}/{epochs:02d} | "
            f"lr={current_lr:.6f} | "
            f"train_loss={avg_train_loss:.4f} | "
            f"val_loss={avg_val_loss:.4f}"
        )

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss

            torch.save(
                {
                    "epoch": epoch,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "best_val_loss": best_val_loss,
                    "history": history,
                    "num_classes": num_classes,
                    "ignore_index": ignore_index,
                    "model_name": model_name,
                },
                save_path,
            )

    print(
        f"Training completed | "
        f"Best Val Loss: {best_val_loss:.4f} | "
        f"Best model saved: {save_path}"
    )

    return model, history

## Eval

In [15]:
def evaluate_model(
    model,
    dataloader,
    device,
    num_classes,
    ignore_index=255,
    criterion=None,
):
    model.eval()

    if criterion is None:
        criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)

    total_loss = 0.0

    metrics = SegmentationMetrics(
        num_classes=num_classes,
        ignore_index=ignore_index
    )

    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].to(device, non_blocking=True)
            masks = batch["mask"].to(device, non_blocking=True)

            outputs = model(images)
            logits = outputs["out"]

            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(
                    logits,
                    size=masks.shape[-2:],
                    mode="bilinear",
                    align_corners=False
                )

            loss = criterion(logits, masks)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            metrics.update(preds, masks)

    avg_loss = total_loss / len(dataloader)
    results = metrics.compute()

    eval_results = {
        "loss": avg_loss,
        "mIoU": results["mIoU"],
        "Pixel Accuracy": results["Pixel Accuracy"],
        "Class IoU": results["Class IoU"]
    }

    return eval_results

In [16]:
def segmentation_report(eval_results, class_names):
    class_iou = eval_results["Class IoU"]

    lines = []
    lines.append(f"{'class':>20s} {'iou':>10s}")
    lines.append("")

    for class_name, iou in zip(class_names, class_iou):
        lines.append(f"{class_name:>20s} {iou:10.4f}")

    lines.append("")
    lines.append(f"{'loss':>20s} {eval_results['loss']:10.4f}")
    lines.append(f"{'mIoU':>20s} {eval_results['mIoU']:10.4f}")
    lines.append(f"{'pixel accuracy':>20s} {eval_results['Pixel Accuracy']:10.4f}")

    return "\n".join(lines)

## Plot

In [17]:
def plot_history(history, title=""):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="Train Loss", marker="o")
    plt.plot(epochs, history["val_loss"], label="Validation Loss", marker="o")
    plt.title(f"Loss {('- ' + title) if title else ''}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

# Main

## Deeplabv3 Resnet50

In [18]:
model = build_deeplabv3_resnet50(NUM_CLASSES)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth


100%|██████████| 161M/161M [00:02<00:00, 64.0MB/s]


Loaded pretrained DeepLabV3-ResNet50 weights.


## Train

In [ ]:
model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX,
    epochs=EPOCHS,
    lr=RESNET_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    checkpoint=None,
    save_path=RESNET_PATH,
)

## Plot

In [ ]:
plot_history(
    history,
    title="DeepLabV3-ResNet50"
)

## Load Checkpoint

In [ ]:
checkpoint = torch.load(RESNET_PATH, map_location=device)

model.load_state_dict(checkpoint["model"])
model = model.to(device)
model.eval()

print("Loaded best checkpoint")
print("Best epoch:", checkpoint["epoch"] + 1)
print("Best val loss:", checkpoint["best_val_loss"])

## Eval

In [ ]:
eval_results = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device,
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX
)

In [ ]:
report_text = segmentation_report(
    eval_results=eval_results,
    class_names=CITYSCAPES_CLASSES
)

print(report_text)

# Test Image

In [ ]:
def denormalize_image(image_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    image = image_tensor.cpu() * std + mean
    image = image.clamp(0, 1)

    image = image.permute(1, 2, 0).numpy()

    return image

In [ ]:
def decode_segmentation_mask(mask):
    h, w = mask.shape

    color_mask = np.zeros((h, w, 3), dtype=np.uint8)

    for class_id in range(NUM_CLASSES):
        color_mask[mask == class_id] = CITYSCAPES_COLORS[class_id]

    color_mask[mask == IGNORE_INDEX] = [0, 0, 0]

    return color_mask

In [ ]:
def predict_single_image(model, sample, device):
    model.eval()

    image = sample["image"]
    mask = sample["mask"]

    input_tensor = image.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        logits = outputs["out"]

        if logits.shape[-2:] != mask.shape[-2:]:
            logits = F.interpolate(
                logits,
                size=mask.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        pred_mask = torch.argmax(logits, dim=1).squeeze(0).cpu()

    return image, mask, pred_mask

In [ ]:
def visualize_predictions(
    model,
    dataset,
    device,
    num_samples=3
):
    random_indices = random.sample(
        range(len(dataset)),
        num_samples
    )

    plt.figure(figsize=(16, 5 * num_samples))

    for row, idx in enumerate(random_indices):
        sample = dataset[idx]

        image, gt_mask, pred_mask = predict_single_image(
            model=model,
            sample=sample,
            device=device
        )

        image_np = denormalize_image(image)

        gt_mask_np = gt_mask.cpu().numpy()
        pred_mask_np = pred_mask.cpu().numpy()

        gt_color = decode_segmentation_mask(gt_mask_np)
        pred_color = decode_segmentation_mask(pred_mask_np)

        plt.subplot(num_samples, 4, row * 4 + 1)
        plt.imshow(image_np)
        plt.title("Input Image")
        plt.axis("off")

        plt.subplot(num_samples, 4, row * 4 + 2)
        plt.imshow(gt_color)
        plt.title("Ground Truth")
        plt.axis("off")

        plt.subplot(num_samples, 4, row * 4 + 3)
        plt.imshow(pred_color)
        plt.title("Prediction")
        plt.axis("off")

        plt.subplot(num_samples, 4, row * 4 + 4)
        plt.imshow(image_np)
        plt.imshow(pred_color, alpha=0.5)
        plt.title("Overlay")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
visualize_predictions(
    model=model,
    dataset=test_dataset,
    device=device,
    num_samples=3
)